In [16]:
# ============================================================
# CELL 1 — CONFIG
# Every weight, threshold, and rule lives here.
# Tune the pipeline by changing ONLY this cell — nothing below
# should need editing once the logic is written.
# ============================================================

CONFIG = {

    # ---- Scoring weights (must sum to 1.0) ----
    "weights": {
        "career_relevance": 0.35,   # does past role-fit actually match the JD's real demands
        "production_signal": 0.25,  # has this person shipped/owned real systems
        "trajectory": 0.20,         # has seniority/responsibility grown over time
        "skill_match": 0.15,        # capped low deliberately — this is the challenge's known trap
        "hireability": 0.05,        # platform behavioral signal (recruiter_response_rate, interview_completion_rate)
    },

    # ---- Consistency penalty (modifier on skill_match, not its own weight) ----
    "consistency_penalty": {
        "enabled": True,
        "gap_threshold": 20,        # point gap between self-rated proficiency and skill_assessment_scores
        "mild_penalty_multiplier": 0.9,   # gap >= threshold but < severe_gap
        "severe_gap_threshold": 35,
        "severe_penalty_multiplier": 0.5, # gap >= severe_gap_threshold
    },

    # ---- Honeypot gate thresholds (pass/fail, applied before scoring) ----
    "honeypot_rules": {
        "max_plausible_age_for_experience": True,
        "min_working_age": 18,
        "max_working_age": 70,
        "check_overlapping_roles": True,
        "check_degree_year_consistency": True,
        "max_allowed_simultaneous_current_roles": 1,
    },

    # ---- Proficiency label -> numeric mapping (for consistency check) ----
    "proficiency_scale": {
        "beginner": 25,
        "intermediate": 55,
        "advanced": 85,
    },

    # ---- Output ----
    "top_n": 100,
    "output_columns": [ "rank","candidate_id", "score", "reasoning"],
}

# Sanity check: weights must sum to 1.0
_weight_sum = round(sum(CONFIG["weights"].values()), 4)
assert _weight_sum == 1.0, f"Weights must sum to 1.0, got {_weight_sum}"

print("Config loaded. Weights sum to:", _weight_sum)
for k, v in CONFIG["weights"].items():
    print(f"  {k}: {v}")

Config loaded. Weights sum to: 1.0
  career_relevance: 0.35
  production_signal: 0.25
  trajectory: 0.2
  skill_match: 0.15
  hireability: 0.05


In [17]:
# ============================================================
# CELL 2 — IMPORTS + DATA LOADING (with upload-completion guard)
# Reads all candidates from a local path. Robust to malformed lines.
# NEW: checks the file is fully written (size stable) before loading,
# so you never accidentally rank a half-uploaded file.
# ============================================================

import json
import gzip
import csv
import time
import io
import os
from datetime import datetime

CANDIDATES_PATH = "/content/candidates.jsonl"     # change if your filename differs
JD_PATH         = "/content/job_description.txt"   # point this at your JD file


def _open_path(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")


def _base_name(path):
    p = path[:-3] if path.endswith(".gz") else path
    return p.lower()


def wait_until_file_stable(path, checks=3, interval=4):
    """Returns once the file size stops changing across `checks` consecutive
    reads `interval` seconds apart -- i.e. the upload/write has finished.
    Prevents loading a partially-uploaded file."""
    if not os.path.exists(path):
        raise RuntimeError(f"{path} does not exist. Upload it first.")
    last = -1
    stable_count = 0
    while stable_count < checks:
        size = os.path.getsize(path)
        if size == last:
            stable_count += 1
        else:
            stable_count = 0
            # print(f"  ... uploading: {size/1e6:.1f} MB (waiting for it to stabilize)")
        last = size
        if stable_count < checks:
            time.sleep(interval)
    print(f"  File stable at {last/1e6:.1f} MB -- upload complete.")
    return last


def load_candidates_from_path(path):
    """Reads all candidates. For .jsonl, parses each line independently and
    skips (with a warning) any malformed line rather than crashing."""
    base = _base_name(path)
    t0 = time.time()
    records = []
    bad_lines = []

    if base.endswith(".jsonl"):
        with _open_path(path) as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    bad_lines.append((line_num, str(e)))
    elif base.endswith(".json"):
        with _open_path(path) as f:
            parsed = json.load(f)
        records = parsed if isinstance(parsed, list) else [parsed]
    elif base.endswith(".csv"):
        with _open_path(path) as f:
            records = [dict(row) for row in csv.DictReader(f)]
    else:
        with _open_path(path) as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    bad_lines.append((line_num, str(e)))

    print(f"Loaded {len(records):,} candidate records from {path} in {time.time()-t0:.2f}s")
    if bad_lines:
        print(f"  WARNING: skipped {len(bad_lines)} malformed line(s). First few:")
        for ln, err in bad_lines[:3]:
            print(f"    line {ln}: {err}")
    if not records:
        raise RuntimeError(f"{path} parsed to 0 records -- check the file.")
    CONFIG["candidates_file_name"] = path
    return records


def load_jd_from_path(path):
    base = _base_name(path)
    with _open_path(path) as f:
        text = f.read()
    if base.endswith(".json"):
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            text = parsed.get("description") or parsed.get("job_description") or text
    if not text or not text.strip():
        raise RuntimeError(f"{path} parsed to empty JD text -- check the file.")
    CONFIG["jd_file_name"] = path
    return text


# ---- Run ----
print("Checking candidates file is fully uploaded...")
wait_until_file_stable(CANDIDATES_PATH)   # blocks until upload done

all_candidates = load_candidates_from_path(CANDIDATES_PATH)

if os.path.exists(JD_PATH):
    jd_text = load_jd_from_path(JD_PATH)
else:
    print(f"NOTE: {JD_PATH} not found -- using inline JD text. Point JD_PATH at your file to override.")
    jd_text = (
        "Senior AI Engineer - Founding Team, Redrob AI. "
        "Own AI/ML systems end-to-end in production. Experience shipping and owning "
        "real ML/AI products is critical. Skills in NLP, LLMs, and modern AI tooling "
        "are valued, but demonstrated ownership, production judgment, and career "
        "trajectory matter far more than a long list of keywords."
    )

_first_candidate = all_candidates[0]

print(f"\nJD loaded ({len(jd_text)} chars)")
print(f"First candidate: {_first_candidate.get('candidate_id', '<no id field>')}")
print(f"Total candidates in memory: {len(all_candidates):,}")

Checking candidates file is fully uploaded...
  File stable at 487.3 MB -- upload complete.
Loaded 100,000 candidate records from /content/candidates.jsonl in 9.41s
NOTE: /content/job_description.txt not found -- using inline JD text. Point JD_PATH at your file to override.

JD loaded (326 chars)
First candidate: CAND_0000001
Total candidates in memory: 100,000


In [18]:
# ============================================================
# CELL 3 — HONEYPOT GATE (revised: fewer false positives)
# Runs BEFORE any scoring. Fails -> candidate dropped entirely.
#
# KEY FIX vs first version: the age/experience check was flagging
# legitimate candidates who simply didn't list their early jobs (stated
# experience naturally exceeds the span of LISTED jobs). That's normal,
# not a honeypot. We now only flag TRULY IMPOSSIBLE cases.
# ============================================================

from datetime import datetime

DATE_FMT = "%Y-%m-%d"


def _parse_date(date_str):
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str, DATE_FMT)
    except (ValueError, TypeError):
        return None


def check_experience_impossible(candidate, rules):
    """Flags experience that is physically IMPOSSIBLE, not merely larger
    than the listed jobs."""
    reasons = []
    profile = candidate.get("profile", {}) or {}
    years_exp = profile.get("years_of_experience")
    if years_exp is None:
        return reasons

    # Hard upper bound: no one has 60+ years of professional experience
    if years_exp > 60:
        reasons.append(f"years_of_experience ({years_exp}) exceeds a plausible human maximum")
        return reasons

    # If degree completion known, check experience doesn't imply working as a child
    education = candidate.get("education", []) or []
    degree_end_years = [e.get("end_year") for e in education if e.get("end_year")]
    if degree_end_years:
        earliest_degree_end = min(degree_end_years)
        now_year = datetime.now().year
        implied_work_start_year = now_year - years_exp
        # Allow working up to ~6 years before degree completion
        if implied_work_start_year < earliest_degree_end - 6:
            reasons.append(
                f"years_of_experience ({years_exp}) implies work starting in "
                f"~{implied_work_start_year:.0f}, more than 6 years before degree "
                f"completion ({earliest_degree_end}) -- implausible"
            )
    return reasons


def check_overlapping_roles(candidate, rules):
    """Flags two non-current roles with overlapping date ranges."""
    reasons = []
    history = candidate.get("career_history", []) or []
    intervals = []
    for h in history:
        if h.get("is_current"):
            continue
        start = _parse_date(h.get("start_date"))
        end = _parse_date(h.get("end_date"))
        if start and end:
            intervals.append((start, end, h.get("company", "unknown")))

    intervals.sort(key=lambda x: x[0])
    for i in range(len(intervals) - 1):
        cur_start, cur_end, cur_company = intervals[i]
        next_start, next_end, next_company = intervals[i + 1]
        overlap_days = (cur_end - next_start).days
        if overlap_days > 31:  # require real overlap, not date-rounding noise
            reasons.append(
                f"overlapping roles: {cur_company} ends {cur_end.date()} but "
                f"{next_company} starts {next_start.date()} ({overlap_days} days overlap)"
            )
    return reasons


def check_multiple_current_roles(candidate, rules):
    """Flags more than the allowed number of simultaneous is_current roles."""
    reasons = []
    history = candidate.get("career_history", []) or []
    current_count = sum(1 for h in history if h.get("is_current"))
    max_allowed = rules.get("max_allowed_simultaneous_current_roles", 1)
    if current_count > max_allowed:
        reasons.append(f"{current_count} simultaneous current roles (max allowed: {max_allowed})")
    return reasons


def check_degree_after_career(candidate, rules):
    """Flags the impossible case where the latest job ends before the degree
    even started. Does NOT flag normal internships-during-study."""
    reasons = []
    education = candidate.get("education", []) or []
    history = candidate.get("career_history", []) or []
    if not education or not history:
        return reasons

    degree_start_years = [e.get("start_year") for e in education if e.get("start_year")]
    if not degree_start_years:
        return reasons
    earliest_degree_start = min(degree_start_years)

    end_dates = []
    for h in history:
        if h.get("is_current"):
            end_dates.append(datetime.now())
        else:
            d = _parse_date(h.get("end_date"))
            if d:
                end_dates.append(d)
    if not end_dates:
        return reasons
    latest_job_end = max(end_dates)

    if latest_job_end.year < earliest_degree_start - 1:
        reasons.append(
            f"latest job ends {latest_job_end.year} but degree only starts "
            f"{earliest_degree_start} -- impossible timeline"
        )
    return reasons


def run_honeypot_gate(candidate, config):
    """Returns (is_valid, all_reasons). is_valid=False means DROP before scoring."""
    rules = config["honeypot_rules"]
    all_reasons = []

    all_reasons += check_experience_impossible(candidate, rules)
    if rules.get("check_overlapping_roles"):
        all_reasons += check_overlapping_roles(candidate, rules)
    all_reasons += check_multiple_current_roles(candidate, rules)
    all_reasons += check_degree_after_career(candidate, rules)

    is_valid = len(all_reasons) == 0
    return is_valid, all_reasons


# ---- Quick test ----
_realistic = {
    "profile": {"years_of_experience": 10.0},
    "career_history": [
        {"start_date": "2022-01-01", "end_date": None, "is_current": True, "duration_months": 42},
        {"start_date": "2019-06-01", "end_date": "2021-12-01", "is_current": False, "duration_months": 30},
    ],
    "education": [{"end_year": 2016, "start_year": 2012}],
}
_impossible = {
    "profile": {"years_of_experience": 15.0},
    "career_history": [{"start_date": "2023-01-01", "end_date": None, "is_current": True, "duration_months": 12}],
    "education": [{"end_year": 2023, "start_year": 2019}],
}
print("Realistic (10y exp, listed 7y):", run_honeypot_gate(_realistic, CONFIG))
print("Impossible (15y exp, degree 2023):", run_honeypot_gate(_impossible, CONFIG))

Realistic (10y exp, listed 7y): (True, [])
Impossible (15y exp, degree 2023): (False, ['years_of_experience (15.0) implies work starting in ~2011, more than 6 years before degree completion (2023) -- implausible'])


In [19]:
# ============================================================
# CELL 4 — BM25 SETUP (replaces TF-IDF/embeddings)
# BM25 is the ranking function search engines use (Elasticsearch etc).
# vs TF-IDF it handles document length and term saturation better.
# Same CPU/offline/seconds profile. No model download.
# ============================================================

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank-bm25"], check=True)
    from rank_bm25 import BM25Okapi

import re
import numpy as np

PRODUCTION_REFERENCE_TEXT = (
    "Owned and shipped production systems end to end. Deployed and scaled "
    "services serving real traffic. On-call ownership of reliability and "
    "incidents. Took systems from design to production."
)

_STOP = {"the","and","for","are","but","with","you","your","our","who","that","this",
         "have","has","not","will","can","far","more","than","long","list","about",
         "into","real","want","looking","team","a","an","of","to","in","on","is","as"}

def _tokenize(text):
    toks = re.findall(r"[a-zA-Z][a-zA-Z+#.]{1,}", (text or "").lower())
    return [t for t in toks if t not in _STOP]


BM25_INDEX = None
JD_TOKENS = None
PROD_REF_TOKENS = None


def fit_bm25(jd_text, all_texts):
    """Build a BM25 index over all candidate docs; populate globals."""
    global BM25_INDEX, JD_TOKENS, PROD_REF_TOKENS
    tokenized_docs = [_tokenize(t) for t in all_texts]
    BM25_INDEX = BM25Okapi(tokenized_docs)
    JD_TOKENS = _tokenize(jd_text)
    PROD_REF_TOKENS = _tokenize(PRODUCTION_REFERENCE_TEXT)


def bm25_scores_vs_jd():
    return np.asarray(BM25_INDEX.get_scores(JD_TOKENS), dtype=float)


def bm25_scores_vs_prodref():
    return np.asarray(BM25_INDEX.get_scores(PROD_REF_TOKENS), dtype=float)


print("BM25 setup loaded. No model download / no GPU / no internet -- pure CPU.")

BM25 setup loaded. No model download / no GPU / no internet -- pure CPU.


In [20]:
# ============================================================
# CELL 5 — SUB-SCORE 1: CAREER RELEVANCE (weight 0.35, the heaviest)
# Measures how well a candidate's ACTUAL career history matches what
# the JD needs -- using embeddings, not keyword overlap. This is the
# signal that catches the challenge's core trap: someone whose skills
# LIST looks AI-heavy but whose real jobs were data engineering should
# score LOW here, because we compare the work they actually DID.
#
# Design choices (all defensible in Stage 5):
#   1. We embed each role's title+description, compare to the JD vector.
#   2. More recent roles count more (recency weighting).
#   3. Longer roles count more (duration weighting).
#   4. Scores are relative across candidates, so we DON'T hard-threshold
#      raw cosine values; the weighting + final ranking sort it out.
# ============================================================

def _role_text(role):
    title = role.get("title", "") or ""
    desc = role.get("description", "") or ""
    return f"{title}. {desc}".strip()


def _recency_weight(role, all_roles):
    """Newest role gets weight 1.0, oldest ~0.4, linear decay."""
    def _start_key(r):
        d = _parse_date(r.get("start_date"))
        return d if d is not None else datetime.now()

    ordered = sorted(all_roles, key=_start_key, reverse=True)
    n = len(ordered)
    if n == 0:
        return 0.0
    try:
        idx = ordered.index(role)
    except ValueError:
        idx = n - 1
    if n == 1:
        return 1.0
    return 1.0 - (0.6 * idx / (n - 1))


def _duration_weight(role):
    """Longer roles = stronger evidence. 36+ months saturates at full weight."""
    months = role.get("duration_months") or 0
    if months <= 0:
        return 0.5
    return 0.3 + 0.7 * min(months, 36) / 36.0


def score_career_relevance(candidate, jd_embedding):
    """Returns (score_0_to_100, evidence_dict). Per-role embedding cosine vs JD,
    weighted by recency * duration, averaged."""
    history = candidate.get("career_history", [])
    if not history:
        return 0.0, {"reason": "no career history", "per_role": []}

    role_texts = [_role_text(r) for r in history]
    role_vecs = embed_texts(role_texts)

    per_role = []
    weighted_sum = 0.0
    weight_total = 0.0

    for role, vec in zip(history, role_vecs):
        sim_score = cosine_to_score(cosine_sim(vec, jd_embedding))
        w = _recency_weight(role, history) * _duration_weight(role)
        weighted_sum += sim_score * w
        weight_total += w
        per_role.append({
            "company": role.get("company"),
            "title": role.get("title"),
            "relevance": round(sim_score, 1),
            "weight": round(w, 3),
        })

    final = weighted_sum / weight_total if weight_total > 0 else 0.0
    return final, {"weighted_relevance": round(final, 1), "per_role": per_role}

In [21]:
# ============================================================
# CELL 6 — PRODUCTION / OWNERSHIP LANGUAGE (used by the BM25 pipeline)
# Provides OWNERSHIP_VERBS, HEDGE_WORDS, and _ownership_language_score.
# Embedding-based scoring removed -- the BM25 pipeline handles the
# production-context similarity itself (via BM25 vs the production
# reference text), so this cell only supplies the lexical ownership signal.
#
# "Ownership" is carried by ACTION VERBS ("owned", "led", "shipped",
# "deployed") and undercut by HEDGE words ("exposure", "assisted",
# "familiar with"). This lexical signal distinguishes "I OWNED and
# SHIPPED this" from "I was EXPOSED to / ASSISTED WITH this."
# ============================================================

OWNERSHIP_VERBS = [
    "owned", "own ", "led", "leading", "built and shipped", "shipped",
    "deployed", "architected", "designed and built", "drove", "spearheaded",
    "on-call", "on call", "end to end", "end-to-end", "from scratch",
    "took ownership", "responsible for", "maintained", "scaled",
]

HEDGE_WORDS = [
    "exposure", "exposed to", "assisted", "supported the", "helped the",
    "familiar with", "worked closely with", "adjacent", "some exposure",
    "involved in", "contributed to", "learning", "building competence",
    "interested in", "transitioning",
]

# The production reference text BM25 scores candidates against (used by Cell 11)
PRODUCTION_REFERENCE_TEXT = (
    "Owned and shipped production systems end to end. Deployed and scaled "
    "services serving real traffic. On-call ownership of reliability and "
    "incidents. Took systems from design to production."
)


def _ownership_language_score(text):
    """Ownership verbs add, hedge words subtract (weighted 1.5x as a strong
    negative). Returns 0..100; neutral baseline is 40."""
    if not text:
        return 0.0
    tl = text.lower()
    own_hits = sum(1 for v in OWNERSHIP_VERBS if v in tl)
    hedge_hits = sum(1 for h in HEDGE_WORDS if h in tl)
    net = own_hits - 1.5 * hedge_hits
    score = 40 + 15 * net
    return float(max(0.0, min(100.0, score)))


print("Ownership-language signal loaded (lexical, no embeddings).")

Ownership-language signal loaded (lexical, no embeddings).


In [22]:
# ============================================================
# CELL 7 — SUB-SCORE 3: TRAJECTORY / SENIORITY PROGRESSION (weight 0.20)
# Has responsibility grown over time, or is the candidate stagnant?
# A senior founding-team role rewards demonstrated growth.
#
# IMPORTANT design guard (discussed with the team):
#   We do NOT naively punish every "downward" move. Two real cases we
#   deliberately do NOT penalize: a manager moving back to senior IC
#   (often a STRONG signal for a hands-on founding engineer), and a
#   career gap. We reward clear upward progression, treat lateral/IC-shift
#   as neutral, and only the genuinely stagnant score low.
# ============================================================

SENIORITY_LEVELS = [
    (6, ["chief", "cto", "vp ", "vice president", "head of", "director"]),
    (5, ["principal", "staff", "distinguished"]),
    (4, ["lead", "manager", "founding", "founder"]),
    (3, ["senior", "sr.", "sr "]),
    (2, ["engineer", "developer", "scientist", "analyst", "specialist"]),
    (1, ["junior", "jr.", "jr ", "intern", "trainee", "associate", "graduate"]),
]


def _title_seniority(title):
    """Maps a job title to a seniority level 1..6. Defaults to 2 (mid IC)."""
    if not title:
        return 2
    tl = title.lower()
    for level, keywords in SENIORITY_LEVELS:
        if any(k in tl for k in keywords):
            return level
    return 2


def score_trajectory(candidate):
    """Returns (score_0_to_100, evidence). Blends peak seniority reached
    with growth from first to latest role. Downward moves only mildly
    penalized so manager->IC shifts aren't unfairly punished."""
    history = candidate.get("career_history", [])
    if not history:
        return 0.0, {"reason": "no career history"}

    def _start_key(r):
        d = _parse_date(r.get("start_date"))
        return d if d is not None else datetime.min
    ordered = sorted(history, key=_start_key)

    levels = [_title_seniority(r.get("title")) for r in ordered]
    first_level, last_level, peak_level = levels[0], levels[-1], max(levels)
    net_growth = last_level - first_level

    peak_score = (peak_level - 1) / 5.0 * 100.0

    if net_growth >= 2:
        growth_score = 100.0
    elif net_growth == 1:
        growth_score = 80.0
    elif net_growth == 0:
        growth_score = 40.0 + (last_level - 1) / 5.0 * 40.0
    else:
        growth_score = max(35.0, peak_score - 25.0)

    final = 0.5 * peak_score + 0.5 * growth_score
    return final, {
        "title_levels_oldest_to_newest": levels,
        "net_growth": net_growth,
        "peak_score": round(peak_score, 1),
        "growth_score": round(growth_score, 1),
        "trajectory": round(final, 1),
    }

In [23]:
# ============================================================
# CELL 8 — SUB-SCORE 4: SKILL MATCH + CONSISTENCY PENALTY (weight 0.15)
# Deliberately the LOWEST weight: the challenge brief explicitly warns
# that ranking by skill-keyword density is the trap. We include skills
# because genuine overlap has SOME signal, but cap it low and -- crucially
# -- apply a CONSISTENCY PENALTY:
#
#   If a candidate self-rates a skill "advanced" but the platform's own
#   skill_assessment_scores measure them much lower, we DON'T trust the
#   self-rating. This catches profiles like Ira Vora -- AI-heavy skills
#   list, but mediocre measured assessments.
#
# Most teams read the skills array at face value. Using the independent
# skill_assessment_scores to discount inflated self-ratings is a strong
# Stage 5 differentiator (a field most teams ignore).
# ============================================================

def _skills_text(candidate):
    skills = candidate.get("skills", []) or []
    skill_names = [s.get("name", "") for s in skills if s.get("name")]
    summary = (candidate.get("profile", {}) or {}).get("summary", "") or ""
    return (summary + ". Skills: " + ", ".join(skill_names)).strip()


def _consistency_multiplier(candidate, config):
    """Compares self-rated proficiency to measured skill_assessment_scores.
    Big gaps -> penalty multiplier < 1.0. Returns (multiplier, evidence)."""
    cfg = config["consistency_penalty"]
    if not cfg.get("enabled"):
        return 1.0, []

    scale = config["proficiency_scale"]
    signals = candidate.get("redrob_signals", {}) or {}
    measured = signals.get("skill_assessment_scores", {}) or {}
    skills = candidate.get("skills", []) or []

    gap_threshold = cfg["gap_threshold"]
    severe_gap = cfg["severe_gap_threshold"]
    mild_mult = cfg["mild_penalty_multiplier"]
    severe_mult = cfg["severe_penalty_multiplier"]

    penalties, evidence = [], []
    for s in skills:
        name = s.get("name")
        prof = (s.get("proficiency") or "").lower()
        if name in measured and prof in scale:
            self_rated = scale[prof]
            measured_val = measured[name]
            gap = self_rated - measured_val
            if gap >= severe_gap:
                penalties.append(severe_mult)
                evidence.append(f"{name}: self={prof}({self_rated}) vs measured={measured_val:.0f} (severe gap {gap:.0f})")
            elif gap >= gap_threshold:
                penalties.append(mild_mult)
                evidence.append(f"{name}: self={prof}({self_rated}) vs measured={measured_val:.0f} (gap {gap:.0f})")

    if not penalties:
        return 1.0, evidence
    return min(penalties), evidence  # most severe penalty, no stacking


def score_skill_match(candidate, jd_embedding, config):
    """Returns (score_0_to_100, evidence). Base = semantic similarity of
    skills+summary to JD, then discounted by the consistency multiplier."""
    skills_text = _skills_text(candidate)
    skills_vec = embed_one(skills_text)
    base = cosine_to_score(cosine_sim(skills_vec, jd_embedding))

    mult, penalty_evidence = _consistency_multiplier(candidate, config)
    final = base * mult

    return final, {
        "base_skill_match": round(base, 1),
        "consistency_multiplier": round(mult, 2),
        "penalties": penalty_evidence,
        "final": round(final, 1),
    }

In [24]:
# ============================================================
# CELL 9 — SUB-SCORE 5: HIREABILITY SIGNAL (weight 0.05, smallest)
# Uses the platform's own BEHAVIORAL data from redrob_signals -- a field
# most teams never open. This doesn't measure skill; it measures whether
# the person is actually engaged and follows through:
#   - recruiter_response_rate: do they respond when recruiters reach out?
#   - interview_completion_rate: do they finish interviews they start?
#   - offer_acceptance_rate: do they accept offers (vs ghosting)?
#
# Weight is deliberately tiny (0.05) -- a tie-breaker, NOT a primary ranker.
# A great engineer with a low response rate shouldn't be sunk by this.
#
# Guard: only used when PRESENT. Missing behavioral data returns a neutral
# 50, so candidates aren't penalized for a field they didn't populate.
# ============================================================

def score_hireability(candidate):
    """Returns (score_0_to_100, evidence). Averages available behavioral
    rates; neutral 50 if none present."""
    signals = candidate.get("redrob_signals", {}) or {}

    components = {
        "recruiter_response_rate": 0.40,
        "interview_completion_rate": 0.35,
        "offer_acceptance_rate": 0.25,
    }

    weighted_sum = 0.0
    weight_used = 0.0
    present = {}
    for field, w in components.items():
        val = signals.get(field)
        if isinstance(val, (int, float)):
            weighted_sum += (val * 100.0) * w
            weight_used += w
            present[field] = val

    if weight_used == 0.0:
        return 50.0, {"reason": "no behavioral signals present", "neutral": True}

    final = weighted_sum / weight_used
    return final, {"present_signals": present, "hireability": round(final, 1)}

In [25]:
# ============================================================
# CELL 10 — MASTER SCORING FUNCTION
# Ties everything together for ONE candidate:
#   1. Run the honeypot gate. Fail -> return None (disqualified).
#   2. Compute all 5 sub-scores.
#   3. Combine with CONFIG weights -> final 0..100.
#   4. Build fact-based reasoning from the actual breakdown -- NO LLM,
#      so it can't hallucinate. Every phrase is grounded in a number.
# ============================================================

def _build_reasoning(candidate, subs, final_score):
    """Concise, honest reasoning from the sub-score breakdown."""
    profile = candidate.get("profile", {}) or {}
    years = profile.get("years_of_experience")
    current_title = profile.get("current_title", "")

    parts = []
    if years is not None and current_title:
        parts.append(f"{years:.0f}y exp, currently {current_title}")
    elif current_title:
        parts.append(f"currently {current_title}")

    cr = subs["career_relevance"]["score"]
    if cr >= 60:   parts.append("strong role-fit to the JD")
    elif cr >= 40: parts.append("moderate role-fit")
    else:          parts.append("limited role-fit (background differs from the role)")

    ps = subs["production_signal"]["score"]
    if ps >= 65:   parts.append("clear production ownership")
    elif ps <= 35: parts.append("mostly supporting/exposure-level work")

    tr = subs["trajectory"]["score"]
    if tr >= 70:   parts.append("upward trajectory")
    elif tr <= 35: parts.append("flat trajectory")

    penalties = subs["skill_match"]["evidence"].get("penalties", [])
    if penalties:  parts.append("self-rated skills exceed measured assessments")

    reasoning = "; ".join(parts)
    return reasoning[0].upper() + reasoning[1:] if reasoning else "Scored on available signals."


def score_candidate(candidate, config, jd_embedding, production_ref_embedding):
    """Full scoring for one candidate. Returns result dict or None if
    disqualified by the honeypot gate."""
    is_valid, honeypot_reasons = run_honeypot_gate(candidate, config)
    if not is_valid:
        return None

    cr_score, cr_ev = score_career_relevance(candidate, jd_embedding)
    ps_score, ps_ev = score_production_signal(candidate, production_ref_embedding)
    tr_score, tr_ev = score_trajectory(candidate)
    sk_score, sk_ev = score_skill_match(candidate, jd_embedding, config)
    hi_score, hi_ev = score_hireability(candidate)

    subs = {
        "career_relevance": {"score": cr_score, "evidence": cr_ev},
        "production_signal": {"score": ps_score, "evidence": ps_ev},
        "trajectory": {"score": tr_score, "evidence": tr_ev},
        "skill_match": {"score": sk_score, "evidence": sk_ev},
        "hireability": {"score": hi_score, "evidence": hi_ev},
    }

    w = config["weights"]
    final_score = (
        w["career_relevance"] * cr_score +
        w["production_signal"] * ps_score +
        w["trajectory"]       * tr_score +
        w["skill_match"]      * sk_score +
        w["hireability"]      * hi_score
    )

    reasoning = _build_reasoning(candidate, subs, final_score)

    return {
        "candidate_id": candidate.get("candidate_id"),
        "score": round(final_score, 2),
        "reasoning": reasoning,
        "breakdown": {
            "career_relevance": round(cr_score, 1),
            "production_signal": round(ps_score, 1),
            "trajectory": round(tr_score, 1),
            "skill_match": round(sk_score, 1),
            "hireability": round(hi_score, 1),
        },
    }

In [26]:
# ============================================================
# CELL 11 — BM25 PIPELINE (percentile norm + technical-domain floor)
# Replaces the TF-IDF pipeline. BM25 = the ranking function search
# engines use; better doc-length handling. Includes:
#   - percentile-normalize BM25 scores (raw BM25 is unbounded)
#   - technical-domain floor: caps career_relevance for non-technical
#     profiles so marketers/accountants can't leak into the top 100
# ============================================================

import numpy as np
import time
import re


def _combined_candidate_text(cand):
    profile = cand.get("profile", {}) or {}
    summary = profile.get("summary", "") or ""
    history = cand.get("career_history", []) or []
    role_bits = [(r.get("title","") or "") + ". " + (r.get("description","") or "") for r in history]
    skills = cand.get("skills", []) or []
    skill_names = ", ".join(s.get("name","") for s in skills if s.get("name"))
    return " ".join([summary] + role_bits + ["Skills: " + skill_names]).strip()


def _jd_keywords(jd_text):
    words = re.findall(r"[a-zA-Z][a-zA-Z+#.]{2,}", jd_text.lower())
    stop = {"the","and","for","are","but","with","you","your","our","who","that",
            "this","have","has","not","will","can","far","more","than","long","list",
            "about","into","real","want","looking","senior","team"}
    return set(w for w in words if w not in stop)


def _skill_overlap_score(cand, jd_kw):
    skills = cand.get("skills", []) or []
    summary = (cand.get("profile", {}) or {}).get("summary", "") or ""
    text = (summary + " " + " ".join(s.get("name","") for s in skills)).lower()
    cand_kw = set(re.findall(r"[a-zA-Z][a-zA-Z+#.]{2,}", text))
    if not jd_kw:
        return 0.0
    return float(min(100.0, len(cand_kw & jd_kw) / 6.0 * 100.0))


def _percentile_normalize(scores):
    """Map raw scores to 0-100 percentile ranks across the pool."""
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    if n <= 1:
        return np.full(n, 50.0)
    return scores.argsort().argsort() / (n - 1) * 100.0


# ---- Non-technical-profile guard (CORRECTED) ----
# Distractor profiles stuff technical buzzwords into their SKILLS list and
# summary, but their actual CAREER-HISTORY work is non-technical (brand design,
# accounting, sales, HR). We judge by what they DID (career descriptions), not
# what they LISTED. This is the challenge's core test.

TECH_ROLE_TERMS = {
    "machine learning","deep learning","ml engineer","ai engineer","data scientist",
    "nlp","computer vision","neural network","trained","fine-tuned","fine-tuning",
    "deployed model","inference","recommendation system","ranking model","llm",
    "model training","ml model","ml models","ml pipeline","data pipeline",
    "built model","machine learning model","predictive model","classifier",
    "research engineer","applied scientist","ml infrastructure","feature engineering",
    "embeddings","transformer","pytorch","tensorflow","scikit","mlops",
    "software engineer","backend engineer","data engineer","ml systems",
}

NONTECH_WORK_TERMS = {
    "brand identity","logo","packaging design","creative direction","typography",
    "month-end close","financial reporting","gaap","tax filings","statutory compliance",
    "accounting","warehouse","fulfillment","enterprise sales","arr quota","quota",
    "stakeholder communication","business diagnostics","process re-engineering",
    "customer support","graphic design","marketing campaign","recruitment","payroll",
}


def _career_only_text(cand):
    h = cand.get("career_history", []) or []
    parts = []
    for r in h:
        parts.append(r.get("title", "") or "")
        parts.append(r.get("description", "") or "")
    return " ".join(parts).lower()


def _is_nontechnical_profile(cand):
    txt = _career_only_text(cand)
    tech = sum(1 for t in TECH_ROLE_TERMS if t in txt)
    nontech = sum(1 for t in NONTECH_WORK_TERMS if t in txt)
    return nontech > tech and tech <= 1



# ---- JD-alignment penalties (encode the JD's explicit "do not want") ----
from datetime import datetime as _dt2
_SERVICES_IND = {"IT Services", "Consulting"}
_SERVICES_CO = {"tcs","infosys","wipro","accenture","cognizant","capgemini","tech mahindra","hcl","ltimindtree"}

def jd_penalty_multiplier(cand, now_str="2026-06-30"):
    mult, notes = 1.0, []
    profile = cand.get("profile", {}) or {}
    sig = cand.get("redrob_signals", {}) or {}
    history = cand.get("career_history", []) or []
    if history:
        all_ind = all(h.get("industry") in _SERVICES_IND for h in history)
        comps = [(h.get("company","") or "").lower() for h in history]
        all_co = bool(comps) and all(any(s in c for s in _SERVICES_CO) for c in comps)
        if all_ind or all_co:
            mult *= 0.45; notes.append("all-services")
    country = profile.get("country", "India")
    if country and country != "India":
        mult *= 0.55; notes.append("non-India")
    non_cur = [h for h in history if not h.get("is_current")]
    if len(non_cur) >= 3:
        avg = sum(h.get("duration_months",0) or 0 for h in non_cur)/len(non_cur)
        if avg < 18: mult *= 0.85; notes.append("job-hopping")
    notice = sig.get("notice_period_days")
    if isinstance(notice,(int,float)) and notice > 90:
        mult *= 0.95; notes.append("long notice")
    rr = sig.get("recruiter_response_rate"); rr = rr if isinstance(rr,(int,float)) else 1.0
    la = sig.get("last_active_date")
    try: di = (_dt2.strptime(now_str,"%Y-%m-%d") - _dt2.strptime(la,"%Y-%m-%d")).days
    except: di = 0
    if rr < 0.15 and di > 90: mult *= 0.5; notes.append("unavailable")
    elif rr < 0.15: mult *= 0.7; notes.append("low response")
    elif di > 150: mult *= 0.88; notes.append("inactive")
    return max(0.3, min(1.0, mult)), notes


def run_bm25_pipeline(candidates, config, jd_text):
    t_start = time.time()

    survivors = []
    disqualified = 0
    for cand in candidates:
        if run_honeypot_gate(cand, config)[0]:
            survivors.append(cand)
        else:
            disqualified += 1
    processed = len(candidates)
    print(f"  honeypot filter: {len(survivors):,} survive, {disqualified:,} dropped "
          f"({100*disqualified/processed:.1f}%, {time.time()-t_start:.1f}s)")

    t_fit = time.time()
    combined_texts = [_combined_candidate_text(c) for c in survivors]
    fit_bm25(jd_text, combined_texts)
    print(f"  BM25 index built on {len(combined_texts):,} docs ({time.time()-t_fit:.1f}s)")

    t_sim = time.time()
    cr_raw = bm25_scores_vs_jd()
    prod_raw = bm25_scores_vs_prodref()
    cr_scores = _percentile_normalize(cr_raw)
    prod_ctx_scores = _percentile_normalize(prod_raw)
    print(f"  BM25 scoring + normalization ({time.time()-t_sim:.1f}s)")

    jd_kw = _jd_keywords(jd_text)
    w = config["weights"]

    scored = []
    for ci, cand in enumerate(survivors):
        cr_score = float(cr_scores[ci])
        # Buzzword-stuffing guard: judge by actual career work, not stuffed skills.
        if _is_nontechnical_profile(cand):
            cr_score = min(cr_score, 10.0)

        history = cand.get("career_history", []) or []
        summary = (cand.get("profile", {}) or {}).get("summary", "") or ""
        prod_text = " ".join([summary] + [(r.get("description","") or "") for r in history])
        lang_score = _ownership_language_score(prod_text)
        ps_score = 0.70 * lang_score + 0.30 * float(prod_ctx_scores[ci])

        tr_score, _ = score_trajectory(cand)

        base_skill = _skill_overlap_score(cand, jd_kw)
        mult, penalties = _consistency_multiplier(cand, config)
        sk_score = base_skill * mult

        hi_score, _ = score_hireability(cand)

        final_score = (
            w["career_relevance"] * cr_score + w["production_signal"] * ps_score +
            w["trajectory"] * tr_score + w["skill_match"] * sk_score +
            w["hireability"] * hi_score
        )
        _jd_mult, _ = jd_penalty_multiplier(cand)
        final_score *= _jd_mult

        subs = {
            "career_relevance": {"score": cr_score, "evidence": {}},
            "production_signal": {"score": ps_score, "evidence": {}},
            "trajectory": {"score": tr_score, "evidence": {}},
            "skill_match": {"score": sk_score, "evidence": {"penalties": penalties}},
            "hireability": {"score": hi_score, "evidence": {}},
        }
        scored.append({
            "candidate_id": cand.get("candidate_id"),
            "score": round(final_score, 2),
            "reasoning": _build_reasoning(cand, subs, final_score),
            "breakdown": {
                "career_relevance": round(cr_score,1), "production_signal": round(ps_score,1),
                "trajectory": round(tr_score,1), "skill_match": round(sk_score,1),
                "hireability": round(hi_score,1),
            },
        })

    scored.sort(key=lambda r: r["score"], reverse=True)
    top = scored[:config["top_n"]]
    for i, r in enumerate(top, 1):
        r["rank"] = i

    elapsed = time.time() - t_start
    stats = {
        "total_processed": processed, "scored": len(scored), "disqualified": disqualified,
        "disqualified_pct": round(100*disqualified/processed,2) if processed else 0,
        "top_n_returned": len(top), "elapsed_seconds": round(elapsed,1),
        "rate_per_sec": round(processed/elapsed,0) if elapsed else 0,
    }
    return top, stats


# ---- Run it ----
print(f"Scoring {len(all_candidates):,} candidates (BM25)...\n")
top_results, run_stats = run_bm25_pipeline(all_candidates, CONFIG, jd_text)
print("\n" + "=" * 50)
for k, v in run_stats.items():
    print(f"  {k}: {v}")
print("\nTop 10:")
for r in top_results[:10]:
    print(f"  #{r['rank']:>3} {r['candidate_id']} score={r['score']:.2f} | {r['reasoning']}")

Scoring 100,000 candidates (BM25)...

  honeypot filter: 97,183 survive, 2,817 dropped (2.8%, 5.2s)
  BM25 index built on 97,183 docs (23.9s)
  BM25 scoring + normalization (2.9s)

  total_processed: 100000
  scored: 97183
  disqualified: 2817
  disqualified_pct: 2.82
  top_n_returned: 100
  elapsed_seconds: 69.4
  rate_per_sec: 1441.0

Top 10:
  #  1 CAND_0088025 score=95.46 | 9y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory
  #  2 CAND_0077337 score=95.22 | 7y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory; self-rated skills exceed measured assessments
  #  3 CAND_0080766 score=94.80 | 9y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory
  #  4 CAND_0061257 score=91.51 | 8y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production own

In [28]:
# ============================================================
# CELL 12 — CSV OUTPUT (submission format)
# Writes the final submission CSV with EXACTLY the required columns
# in the required order: candidate_id, rank, score, reasoning.
# Includes self-validation so you catch a format problem here, not
# after submission.
# ============================================================

import csv as _csv

OUTPUT_PATH = "/content/team_submission.csv"  # rename to your required filename


def _validate_submission(rows, config):
    """Checks rows against the spec: exactly top_n rows, ranks 1..N contiguous,
    score non-increasing, reasoning non-empty. Raises on any problem."""
    top_n = config["top_n"]
    problems = []

    if len(rows) != top_n:
        problems.append(f"expected {top_n} rows, got {len(rows)}")

    ranks = [r["rank"] for r in rows]
    if ranks != list(range(1, len(rows) + 1)):
        problems.append("ranks are not a clean 1..N sequence")

    for i in range(1, len(rows)):
        if rows[i]["score"] > rows[i - 1]["score"]:
            problems.append(
                f"score increases at rank {rows[i]['rank']} "
                f"({rows[i-1]['score']} -> {rows[i]['score']})"
            )
            break

    empty_reasons = [r["rank"] for r in rows if not (r.get("reasoning") or "").strip()]
    if empty_reasons:
        problems.append(f"empty reasoning at ranks: {empty_reasons[:5]}")

    if problems:
        raise ValueError("Submission validation failed:\n  - " + "\n  - ".join(problems))
    return True


def write_submission_csv(top_results, config, output_path=OUTPUT_PATH):
    """Validates then writes ranked results to CSV in required column order."""
    columns = config["output_columns"]
    _validate_submission(top_results, config)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for r in top_results:
            writer.writerow({
                "rank": r["rank"],
                "candidate_id": r["candidate_id"],
                "score": r["score"],
                "reasoning": r["reasoning"],
            })

    print(f"Submission written to {output_path}")
    print(f"  {len(top_results)} rows, columns: {columns}")
    return output_path


# ---- Write it ----
_path = write_submission_csv(top_results, CONFIG)

print("\nFirst 5 rows of the CSV:")
with open(_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i > 5:
            break
        print("  " + line.rstrip())

Submission written to /content/team_submission.csv
  100 rows, columns: ['rank', 'candidate_id', 'score', 'reasoning']

First 5 rows of the CSV:
  rank,candidate_id,score,reasoning
  1,CAND_0088025,95.46,"9y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory"
  2,CAND_0077337,95.22,"7y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory; self-rated skills exceed measured assessments"
  3,CAND_0080766,94.8,"9y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory"
  4,CAND_0061257,91.51,"8y exp, currently Staff Machine Learning Engineer; strong role-fit to the JD; clear production ownership; upward trajectory"
  5,CAND_0068351,90.4,"6y exp, currently Lead AI Engineer; strong role-fit to the JD; clear production ownership; upward trajectory"
